In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langgraph.checkpoint.memory import MemorySaver

In [2]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [3]:
from langgraph.graph import add_messages
from langchain_core.messages import BaseMessage, HumanMessage


class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage],add_messages]

In [4]:
def chat_node(state: ChatState) -> ChatState:
    messages = state["messages"]
    response = llm.invoke(messages)
    return {"messages": [response]}

In [5]:
graph = StateGraph(ChatState)

graph.add_node("chat_node",chat_node)
graph.add_edge(START, "chat_node")
graph.add_edge("chat_node",END)
res = graph.compile(checkpointer=MemorySaver())

In [6]:
from langchain_core.messages import HumanMessage
thread_id = '1'
while True:
    inp = input("Enter message: ")

    if inp.lower() == "quit":
        break
    config = {'configurable':{"thread_id":thread_id}}
    ai_response = res.invoke({"messages": [HumanMessage(content=inp)]},config=config)
    print(ai_response["messages"][-1].content)

How can I assist you today?


KeyboardInterrupt: Interrupted by user